# Step 3: build the eADAGEs

Builds one eADAGE per group from its 25 models with the lab's two scripts: the weighted Pearson correlation between
every pair of nodes (Python), then consensus clustering into 450 clusters and the mean of each cluster (R). eADAGE
biases are zero. Saves `runs/eadage/eadage_group{g}.npz`. Needs about 32 GB of memory; `build_eadage.slurm` builds
one group per job.

Settings.

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's startup messages

PROJECT = os.path.expandvars("$SCRATCH/CSC-488-Code")    # project folder: data, lab code and results
DATA_PATH = os.path.join(PROJECT, "data", "Copy of ecoli_MG1655plus_genome_part2_log_counts_norm_01.csv")
SEQADAGE = os.path.join(PROJECT, "seqADAGE")              # the lab's code (my fork, Jordin branch)
RUN_DIR = os.path.join(PROJECT, "runs")                   # every notebook saves and reads its results here
os.chdir(PROJECT)

N_GROUPS, MODELS_PER_GROUP, BASE_SEED = 5, 25, 171       # 5 groups of 25 models, seeds 171 to 295

import glob
import subprocess
import time
import numpy as np
import pandas as pd

RSCRIPT = os.path.expandvars("$SCRATCH/envs/radage/bin/Rscript")  # R environment with ClusterR, ff and readr
CLUSTER_SEED = 735                       # the R script's clustering seed (as in the lab's batch script)
N_CORES = len(os.sched_getaffinity(0))   # cores for the correlation script
GROUPS = list(range(N_GROUPS))           # or e.g. [0]
FORCE = False                            # True = rebuild eADAGEs that are already saved

# as a batch job, array task g builds only group g
if "SLURM_ARRAY_TASK_ID" in os.environ:
    GROUPS = [int(os.environ["SLURM_ARRAY_TASK_ID"])]

Seeds and file paths.

In [ ]:
def group_seed(g, i):
    """Seed of model i of group g (model number N = 25 g + i has seed 171 + N)."""
    return BASE_SEED + g * MODELS_PER_GROUP + i


def model_path(g, seed):
    return os.path.join(RUN_DIR, "models", f"group{g}", f"model_seed{seed}.npz")


def eadage_path(g):
    return os.path.join(RUN_DIR, "eadage", f"eadage_group{g}.npz")


def load_model(path):
    """A saved model (single model or eADAGE) as a dict of arrays."""
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}


def load_group(g):
    """The 25 models of group g, in seed order."""
    return [load_model(model_path(g, group_seed(g, i))) for i in range(MODELS_PER_GROUP)]

Build one group's eADAGE with the lab's scripts, then read back the eADAGE weights and the cluster of every single-model
node.

In [ ]:
def build_eadage(g, models):
    lab_dir = os.path.join(SEQADAGE, "ensemble_construction")
    weights_dir = os.path.join(RUN_DIR, "models", "lab_format", "weights")
    out_dir = os.path.join(RUN_DIR, "eadage", f"R_group{g}")
    scratch = os.path.join(RUN_DIR, "eadage", "scratch")
    wcor = os.path.join(RUN_DIR, "eadage", f"wcor_group{g}.tsv")  # about 3 GB, deleted at the end
    log_path = os.path.join(RUN_DIR, "eadage", f"lab_scripts_group{g}.log")
    for d in [out_dir, scratch]:
        os.makedirs(d, exist_ok=True)
    seeds = [int(m["seed"]) for m in models]
    G, K = models[0]["W"].shape
    steps = [("weighted Pearson correlations",
              [sys.executable, "weighted_pearson_correlation_parallel_edit_dcc.py", weights_dir, wcor,
               f"{seeds[0]},{seeds[-1]}", str(N_CORES), str(G)]),
             ("consensus clustering",
              [RSCRIPT, "ensemble_construction_p_dcc.R", DATA_PATH, weights_dir + "/", scratch, str(K),
               str(seeds[0]), str(seeds[-1]), str(CLUSTER_SEED), "weighted", out_dir, wcor])]
    with open(log_path, "w") as log:
        for name, cmd in steps:
            print(f"group {g}: {name}...")
            if subprocess.run(cmd, cwd=lab_dir, stdout=log, stderr=subprocess.STDOUT).returncode != 0:
                raise RuntimeError(f"{name} failed, see {log_path}")
    os.remove(wcor)

    # the eADAGE weights, and each single-model node's cluster (R numbers clusters from 1)
    lines = open(glob.glob(os.path.join(out_dir, "*_ADAGE.txt"))[0]).read().split("\n")
    W = np.array([[float(v) for v in l.split("\t")]
                  for l in lines[lines.index("weight matrix") + 1:lines.index("hidden bias vector")]], "float32")
    clusters = pd.read_csv(glob.glob(os.path.join(out_dir, "*_clusters.csv"))[0], index_col=0)["cluster"]
    labels = clusters.loc[[f"seed{s}_node{n + 1}" for s in seeds for n in range(K)]].values - 1
    return dict(W=W, eb=np.zeros(W.shape[1], "float32"), act1=models[0]["act1"], labels=labels,
                node_model=np.repeat(np.arange(len(models)), K), node_idx=np.tile(np.arange(K), len(models)),
                seeds=np.array(seeds))

Build the eADAGEs (skips groups that are already built).

In [ ]:
for g in GROUPS:
    if os.path.exists(eadage_path(g)) and not FORCE:
        print(f"group {g}: eADAGE already built, skipping")
        continue
    start = time.time()
    e = build_eadage(g, load_group(g))
    np.savez(eadage_path(g), **e)
    print(f"group {g}: built in {(time.time() - start) / 60:.0f} min")

Cluster sizes, and the weight magnitude of the nodes in large and other clusters (thesis Section 4.2).

In [ ]:
rows = []
for g in range(N_GROUPS):
    e = load_model(eadage_path(g))
    norm = np.concatenate([np.linalg.norm(m["W"], axis=0) for m in load_group(g)])  # every single-model node
    sizes = np.bincount(e["labels"], minlength=e["W"].shape[1])
    member_norm = np.array([np.median(norm[e["labels"] == c]) if sizes[c] else np.nan for c in range(len(sizes))])
    large = sizes > 100
    rows.append(dict(group=g, nodes=int(sizes.sum()), median_size=np.median(sizes), largest=sizes.max(),
                     single_node=int((sizes == 1).sum()), over_100=int(large.sum()),
                     nodes_in_over_100=int(sizes[large].sum()),
                     weight_magnitude_large=np.nanmedian(member_norm[large]) if large.any() else np.nan,
                     weight_magnitude_other=np.nanmedian(member_norm[~large])))
clusters = pd.DataFrame(rows)
print(f"{clusters['over_100'].sum()} clusters with more than 100 nodes, holding "
      f"{100 * clusters['nodes_in_over_100'].sum() / clusters['nodes'].sum():.0f}% of all nodes")
clusters.round(3)